# 03 · Your First Agent — the loop, live

**Agentic AI for Actuaries** · IFoA Workshop · 10 July 2026 · Hub: `github.com/rohanyashraj/ifoa-workshop`

> All data in this notebook is **hypothetical** — ABC Insurer is a fictional entity calibrated to plausible Indian market experience, for teaching only.

**Used in:** Session 2, Part 1 (Inside the Agent Loop). 
**You will:** build a hello-agent in ~8 lines with one tool, read its ReAct trace, then break it and watch structured error handling keep it graceful.

Prereq: `GOOGLE_API_KEY` in Colab Secrets (as in notebook 01).

In [ ]:
%pip install -q -U agno google-genai

In [ ]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

## §1 · The raw mechanic — what 'function calling' actually is
Before any framework: the model never runs code. It emits a **JSON request**; your runtime decides. Every guardrail you will ever build lives in that gap.

In [ ]:
from google import genai
from IPython.display import Markdown, display

client = genai.Client()

def claim_frequency(region: str) -> float:
    """Earned claim frequency (claims per policy-year) for a region tier, ABC Motor 2024."""
    table = {"Tier1": 0.091, "Tier2": 0.078, "Tier3": 0.069}
    if region not in table:
        raise ValueError(f"Unknown region '{region}'. Valid: {list(table)}")
    return table[region]

# Hand the model the tool schema; it responds with a *request*, not an execution
resp = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="What is the claim frequency in Tier1?",
    config={"tools": [claim_frequency]},   # the SDK auto-builds the schema and runs the loop
)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(resp.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

## §2 · Hello, agent — eight lines with Agno
`show_tool_calls=True` prints the trace — the agent equivalent of an audit trail. Leave it on in development, always.

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini

agent = Agent(
    model=Gemini(id="gemini-3.1-flash-lite"),
    tools=[claim_frequency],
    show_tool_calls=True,
    markdown=True,
)

agent.print_response("Compare claim frequency between Tier1 and Tier3, as a percentage difference.")

**Read your trace.** The model called the tool twice — once per tier — then computed the comparison from *tool results*, not imagination. Nobody told it to call twice: that was the ReAct loop deciding.

## §3 · Break it — structured errors the model can reason about
Ask about a region that doesn't exist. The tool raises; Agno hands the error back **as data**; the model recovers gracefully instead of inventing a number.

In [ ]:
agent.print_response("What is the claim frequency in Tier4?")
# Expected behaviour: the agent reports that Tier4 is not a valid region and lists the valid tiers.
# A stack trace teaches the model nothing; {"error": "Unknown region"} is something it can reason about.

## §4 · Exercises (5 minutes)
1. Add a second tool `average_severity(region: str) -> float` (invent plausible numbers) and ask for **pure premium** by region — watch the agent chain both tools and do the multiplication with tool numbers.
2. Set `show_tool_calls=False`, re-run, and notice what you lose. Turn it back on. That feeling is checklist question 10.
3. Ask something *outside* the tools' competence ('what will frequency be in 2027?') and observe how the agent hedges — or doesn't. What guardrail would you add?

Next: notebook 04 — the real build.